In [3]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from typing import TypedDict
from dotenv import load_dotenv

In [ ]:
load_dotenv()

True

In [5]:
llm=HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V3",
    task="text-generation"
)

model=ChatHuggingFace(llm=llm)

/Users/devgupta/Desktop/LangGraph/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# create a state

class LLMState(TypedDict):

    question: str
    answer: str

In [7]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state


In [10]:
# create our graph

graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [9]:
# execute

intial_state = {'question': 'How far is moon from the earth?'}

final_state = workflow.invoke(intial_state)

print(final_state['answer'])


The average distance from the Earth to the Moon is approximately **384,400 kilometers (238,855 miles)**.  

However, because the Moon orbits the Earth in an elliptical (oval-shaped) path, its distance varies:  
- **Perigee (closest approach):** ~363,300 km (225,623 miles)  
- **Apogee (farthest distance):** ~405,500 km (251,966 miles)  

This variation affects tides and the apparent size of the Moon in the sky.  

Would you like details on how this distance is measured?


In [11]:
model.invoke('How far is moon from the earth?').content

"The average distance from the **Earth to the Moon** is about **384,400 kilometers (238,855 miles)**.  \n\nHowever, because the Moon's orbit is elliptical (not perfectly circular), its distance varies:  \n- **Perigee** (closest approach): ~ **363,300 km (225,700 mi)**  \n- **Apogee** (farthest distance): ~ **405,500 km (252,000 mi)**  \n\nThis variation affects phenomena like **supermoon** (when the Moon is closer and appears larger) and **micromoon** (when it's farther and appears smaller).  \n\nWould you like details on how this distance is measured or historical changes in the Moon's orbit?"